In [34]:
import re
from typing import Optional


def filter_string(
    input_string: str,
    max_consecutive: Optional[int] = None
) -> str:
    """
    Фильтрует строку, оставляя только латиницу, кириллицу, пунктуацию, числа и LaTeX-символы.
    Удаляет символы, которые повторяются больше заданного числа раз подряд.
    
    Args:
        input_string (str): Входная строка для фильтрации
        max_consecutive (int, optional): Максимальное количество одинаковых символов подряд.
                                        Если None, ограничение не применяется.
    
    Returns:
        str: Отфильтрованная строка
    """
    if not input_string:
        return ""
    
    # Основной шаблон для разрешенных символов
    allowed_pattern = r'[a-zA-Zа-яА-ЯёЁ0-9\s\.,!?;:\(\)\[\]\{\}\-\+\=\*\/&%$#@~`\\^_\|«»""\'\'—–‐…]'
    
    # Фильтруем строку, оставляя только разрешенные символы
    filtered_chars = []
    for char in input_string:
        if re.match(allowed_pattern, char):
            filtered_chars.append(char)
    
    # Первый этап фильтрации - базовые символы
    stage1_string = ''.join(filtered_chars)
    
    # Второй этап - удаление повторяющихся символов если задано ограничение
    if max_consecutive is not None and max_consecutive > 0:
        return remove_consecutive_duplicates(stage1_string, max_consecutive)
    
    return stage1_string


def remove_consecutive_duplicates(string: str, max_consecutive: int) -> str:
    """
    Удаляет символы, которые повторяются больше max_consecutive раз подряд.
    
    Args:
        string (str): Входная строка
        max_consecutive (int): Максимальное количество повторений. Если max_consecutive=1, каждый символ может встретиться 1 раз подряд.
    
    Returns:
        str: Строка с удаленными лишними повторениями
    """
    if not string or max_consecutive <= 0:
        return string
    
    result = []
    count = 1
    prev_char = string[0]
    
    for i in range(1, len(string)):
        current_char = string[i]
        
        if current_char == prev_char:
            count += 1
        else:
            # Добавляем символы (не более max_consecutive)
            result.append(prev_char * min(count, max_consecutive))
            count = 1
            prev_char = current_char
    
    # Добавляем последнюю группу символов
    result.append(prev_char * min(count, max_consecutive))
    
    return ''.join(result)


In [35]:
input_string = r"$E = mc^2$ and $\alpha = \beta$«»\'\'—–‐…############"
max_consecutive = 10
result = filter_string(input_string, max_consecutive=max_consecutive)
print(result)

$E = mc^2$ and $\alpha = \beta$«»\'\'—–‐…##########


In [ ]:
def run_comprehensive_tests():
    """
    Комплексное тестирование функции filter_string
    """
    print("=" * 60)
    print("КОМПЛЕКСНОЕ ТЕСТИРОВАНИЕ ФУНКЦИИ FILTER_STRING")
    print("=" * 60)
    
    test_cases = [
        # Базовые тесты
        {
            "name": "Базовые символы",
            "input": r"Hello World 123",
            "max_consecutive": None,
            "expected": r"Hello World 123"
        },
        {
            "name": "Русский текст",
            "input": r"Привет Мир 456",
            "max_consecutive": None,
            "expected": r"Привет Мир 456"
        },
        
        # Тесты с пунктуацией
        {
            "name": "Пунктуация и символы",
            "input": r"Hello! How are you? I'm fine, thanks.",
            "max_consecutive": None,
            "expected": r"Hello! How are you? I'm fine, thanks."
        },
        {
            "name": "Математические символы",
            "input": r"1 + 2 = 3; 4 * 5 = 20; 10 / 2 = 5",
            "max_consecutive": None,
            "expected": r"1 + 2 = 3; 4 * 5 = 20; 10 / 2 = 5"
        },
        
        # Тесты с LaTeX
        {
            "name": "Простой LaTeX",
            "input": r"$E = mc^2$ and $\alpha = \beta$",
            "max_consecutive": None,
            "expected": r"$E = mc^2$ and $\alpha = \beta$"
        },
        {
            "name": "Сложный LaTeX",
            "input": r"$\sum_{i=1}^n i = \frac{n(n+1)}{2}$",
            "max_consecutive": None,
            "expected": r"$\sum_{i=1}^n i = \frac{n(n+1)}{2}$"
        },
        
        # Тесты с повторениями
        {
            "name": "Множественные восклицательные знаки",
            "input": r"Wow!!! So cool!!!!!",
            "max_consecutive": 3,
            "expected": r"Wow!!! So cool!!!"
        },
        {
            "name": "Повторы букв",
            "input": r"Noooooo wayyyyyy sooo many repeatssss",
            "max_consecutive": 2,
            "expected": r"Noo wayy soo many repeatss"
        },
        
        # Тесты с удалением нежелательных символов
        {
            "name": "Эмодзи и символы",
            "input": r"Hello 🚀 World 🌍 Test ✅",
            "max_consecutive": None,
            "expected": r"Hello  World  Test "
        },
        {
            "name": "Китайские иероглифы",
            "input": r"中文 测试 русский text",
            "max_consecutive": None,
            "expected": r"  русский text"
        },
        {
            "name": "Арабский текст",
            "input": r"مرحبا بالعالم hello world",
            "max_consecutive": None,
            "expected": r"  hello world"
        },
        
        # Смешанные тесты
        {
            "name": "Смешанный текст с разными символами",
            "input": r"Hello 世界 Привет 123! @#$%^&*()",
            "max_consecutive": None,
            "expected": r"Hello  Привет 123! @#$%^&*()"
        },
        {
            "name": "Markdown-like текст",
            "input": r"# Header **bold** *italic* `code`",
            "max_consecutive": None,
            "expected": r"# Header **bold** *italic* `code`"
        },
        {
            "name": "URL и email",
            "input": r"Visit https://example.com or email test@test.com",
            "max_consecutive": None,
            "expected": r"Visit https://example.com or email test@test.com"
        },
        
        # Граничные случаи
        {
            "name": "Пустая строка",
            "input": "",
            "max_consecutive": 3,
            "expected": ""
        },
        {
            "name": "Только недопустимые символы",
            "input": "🚀🌍🎉",
            "max_consecutive": None,
            "expected": ""
        },
        {
            "name": "Только пробелы",
            "input": "     ",
            "max_consecutive": 2,
            "expected": "  "
        },
        
        # Сложные случаи с повторениями
        {
            "name": "Разные повторения в одной строке",
            "input": "Yesss!!! Wait... Nooo???? Maybe...",
            "max_consecutive": 2,
            "expected": "Yess!! Wait.. Noo?? Maybe.."
        },
        {
            "name": "Повторы с лимитом 0",
            "input": "Hello!!!",
            "max_consecutive": 0,
            "expected": "Hello!!!"
        },

        # Комбинированный текст
        {
            "name": "Комбинированный текст",
            "input": "Yesرحبا بالعالم ss!!! ;763во中文 测试шаdfhjbib//alpha* /beta = ? #🚀🌍🎉#### ",
            "max_consecutive": 2,
            "expected": "Yes  ss!! ;763во шаdfhjbib//alpha* /beta = ? ## "
        },
    ]
    
    passed_tests = 0
    failed_tests = []
    
    for i, test in enumerate(test_cases, 1):
        print(f"\nТест {i}: {test['name']}")
        print(f"Вход: '{test['input']}'")
        
        # Вызываем функцию с соответствующими параметрами
        result = filter_string(
            test['input'], 
            max_consecutive=test['max_consecutive']
        )
        
        print(f"Ожидалось: '{test['expected']}'")
        print(f"Получилось: '{result}'")
        
        if result == test['expected']:
            print("✅ ПРОЙДЕН")
            passed_tests += 1
        else:
            print("❌ НЕ ПРОЙДЕН")
            failed_tests.append({
                'name': test['name'],
                'input': test['input'],
                'expected': test['expected'],
                'actual': result
            })
    
    # Вывод статистики
    print("\n" + "=" * 60)
    print(f"РЕЗУЛЬТАТЫ ТЕСТИРОВАНИЯ")
    print(f"Пройдено: {passed_tests}/{len(test_cases)}")
    print(f"Провалено: {len(failed_tests)}/{len(test_cases)}")
    
    if failed_tests:
        print("\nПРОВАЛЕННЫЕ ТЕСТЫ:")
        for fail in failed_tests:
            print(f"\n❌ {fail['name']}")
            print(f"   Вход: '{fail['input']}'")
            print(f"   Ожидалось: '{fail['expected']}'")
            print(f"   Получилось: '{fail['actual']}'")


def test_specific_edge_cases():
    """
    Тестирование граничных случаев и особых сценариев
    """
    print("\n" + "=" * 60)
    print("ТЕСТИРОВАНИЕ ГРАНИЧНЫХ СЛУЧАЕВ")
    print("=" * 60)
    
    # Тесты с различными лимитами повторений
    repeat_tests = [
        ("Noooooooooo!!!", 1, "No!"),
        ("Noooooooooo!!!", 3, "Nooo!!!"),
        ("Noooooooooo!!!", 5, "Nooooo!!!"),
        ("Noooooooooo!!!", 10, "Noooooooooo!!!"),
    ]
    
    for i, (input_text, limit, expected) in enumerate(repeat_tests, 1):
        result = filter_string(input_text, max_consecutive=limit)
        status = "✅" if result == expected else "❌"
        print(f"{status} Повторы тест {i}: лимит {limit} '{input_text}' -> '{result}'")


def test_performance():
    """
    Тестирование производительности на больших строках
    """
    print("\n" + "=" * 60)
    print("ТЕСТИРОВАНИЕ ПРОИЗВОДИТЕЛЬНОСТИ")
    print("=" * 60)
    
    import time
    
    # Создаем большую строку для тестирования
    large_text = "Hello! " * 10000 + "A" * 1000 + "!!!" * 500
    mixed_text = "Привет " * 5000 + "$\\alpha$" * 2000 + "🚀" * 1000
    
    # Тест 1: Большой текст без ограничения повторений
    start_time = time.time()
    result1 = filter_string(large_text, max_consecutive=None)
    time1 = time.time() - start_time
    print(f"Большой текст (без ограничений): {len(large_text)} -> {len(result1)} символов за {time1:.4f} сек")
    
    # Тест 2: Большой текст с ограничением повторений
    start_time = time.time()
    result2 = filter_string(large_text, max_consecutive=2)
    time2 = time.time() - start_time
    print(f"Большой текст (с ограничениями): {len(large_text)} -> {len(result2)} символов за {time2:.4f} сек")
    
    # Тест 3: Смешанный текст
    start_time = time.time()
    result3 = filter_string(mixed_text, max_consecutive=3)
    time3 = time.time() - start_time
    print(f"Смешанный текст: {len(mixed_text)} -> {len(result3)} символов за {time3:.4f} сек")


if __name__ == "__main__":
    # Запускаем все тесты
    run_comprehensive_tests()
    test_specific_edge_cases()
    test_performance()
    
    # Дополнительные демонстрационные примеры
    print("\n" + "=" * 60)
    print("ДЕМОНСТРАЦИОННЫЕ ПРИМЕРЫ")
    print("=" * 60)
    
    demo_examples = [
        r"Hello!!! This is a test with 🚀 emoji and 中文 text!",
        r"LaTeX: $\sum_{i=1}^\infty \frac{1}{i^2} = \frac{\pi^2}{6}$",
        r"Много    пробелов    и!!! восклицательных!! знаков???",
    ]
    
    for example in demo_examples:
        print(f"\nИсходный текст: {example}")
        result = filter_string(example, max_consecutive=2)
        print(f"После фильтрации: {result}")

КОМПЛЕКСНОЕ ТЕСТИРОВАНИЕ ФУНКЦИИ FILTER_STRING

Тест 1: Базовые символы
Вход: 'Hello World 123'
Ожидалось: 'Hello World 123'
Получилось: 'Hello World 123'
✅ ПРОЙДЕН

Тест 2: Русский текст
Вход: 'Привет Мир 456'
Ожидалось: 'Привет Мир 456'
Получилось: 'Привет Мир 456'
✅ ПРОЙДЕН

Тест 3: Пунктуация и символы
Вход: 'Hello! How are you? I'm fine, thanks.'
Ожидалось: 'Hello! How are you? I'm fine, thanks.'
Получилось: 'Hello! How are you? I'm fine, thanks.'
✅ ПРОЙДЕН

Тест 4: Математические символы
Вход: '1 + 2 = 3; 4 * 5 = 20; 10 / 2 = 5'
Ожидалось: '1 + 2 = 3; 4 * 5 = 20; 10 / 2 = 5'
Получилось: '1 + 2 = 3; 4 * 5 = 20; 10 / 2 = 5'
✅ ПРОЙДЕН

Тест 5: Простой LaTeX
Вход: '$E = mc^2$ and $\alpha = \beta$'
Ожидалось: '$E = mc^2$ and $\alpha = \beta$'
Получилось: '$E = mc^2$ and $\alpha = \beta$'
✅ ПРОЙДЕН

Тест 6: Сложный LaTeX
Вход: '$\sum_{i=1}^n i = \frac{n(n+1)}{2}$'
Ожидалось: '$\sum_{i=1}^n i = \frac{n(n+1)}{2}$'
Получилось: '$\sum_{i=1}^n i = \frac{n(n+1)}{2}$'
✅ ПРОЙДЕН

Тест 7: Множе